In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf

# RBI Policy Dates
rbi_dates = [
    '2024-02-08',
    '2024-04-05',
    '2024-06-07',
    '2024-08-08',
    '2024-10-09',
    '2024-12-06',
    '2025-02-07',
    '2025-04-09',
]

# Pull Nifty data
nifty = yf.Ticker("^NSEI")
hist = nifty.history(start="2024-01-01", end="2025-06-01")
hist['returns'] = hist['Close'].pct_change()
hist['HV10'] = hist['returns'].rolling(10).std() * np.sqrt(252) * 100

print("Data loaded:", hist.shape)
print(hist.tail(3))

In [ ]:
results = []

for date in rbi_dates:
    rbi_date = pd.Timestamp(date).tz_localize('Asia/Kolkata')

    # Get HV 5 days before and 5 days after
    start = rbi_date - pd.Timedelta(days=10)
    end = rbi_date + pd.Timedelta(days=10)

    window = hist[(hist.index >= start) & (hist.index <= end)]['HV10'].dropna()

    if len(window) >= 4:
        # Find RBI date position
        before_dates = hist[(hist.index >= start) & (hist.index < rbi_date)]['HV10'].dropna()
        after_dates = hist[(hist.index > rbi_date) & (hist.index <= end)]['HV10'].dropna()

        if len(before_dates) >= 2 and len(after_dates) >= 2:
            hv_before = before_dates.iloc[-1]  # HV day before RBI
            hv_after = after_dates.iloc[1]     # HV 2 days after RBI
            hv_change = hv_after - hv_before
            hv_change_pct = (hv_change / hv_before) * 100

            results.append({
                'rbi_date': date,
                'hv_before': round(hv_before, 2),
                'hv_after': round(hv_after, 2),
                'hv_change': round(hv_change, 2),
                'hv_change_pct': round(hv_change_pct, 2),
                'iv_crushed': hv_after < hv_before
            })

results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), facecolor='#0a0a0a')

dates = results_df['rbi_date']
x = range(len(dates))
width = 0.35

# Chart 1 - HV Before vs After
ax1.set_facecolor('#0a0a0a')
ax1.bar([i - width/2 for i in x], results_df['hv_before'],
        width, label='HV Before RBI', color='#FF6B6B', alpha=0.9)
ax1.bar([i + width/2 for i in x], results_df['hv_after'],
        width, label='HV After RBI', color='#4ECDC4', alpha=0.9)
ax1.set_xticks(list(x))
ax1.set_xticklabels(dates, rotation=45, fontsize=9, color='white')
ax1.set_title('HV Before vs After RBI Dates', color='white', fontsize=12, pad=10)
ax1.set_ylabel('Historical Volatility %', color='white')
ax1.tick_params(colors='white')
ax1.legend(facecolor='#1a1a1a', labelcolor='white')
ax1.grid(True, alpha=0.2, color='white')
ax1.spines['bottom'].set_color('#333')
ax1.spines['left'].set_color('#333')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)

# Chart 2 - HV Change %
ax2.set_facecolor('#0a0a0a')
bar_colors = ['#2ECC71' if v < 0 else '#E74C3C'
              for v in results_df['hv_change_pct']]
bars = ax2.bar(list(x), results_df['hv_change_pct'],
               color=bar_colors, alpha=0.9, width=0.5)
ax2.axhline(y=0, color='white', linewidth=1, alpha=0.5)
ax2.set_xticks(list(x))
ax2.set_xticklabels(dates, rotation=45, fontsize=9, color='white')
ax2.set_title('HV Change % After RBI  |  Green = IV Crush  |  Red = IV Spike',
              color='white', fontsize=12, pad=10)
ax2.set_ylabel('HV Change %', color='white')
ax2.tick_params(colors='white')
ax2.grid(True, alpha=0.2, color='white')
ax2.spines['bottom'].set_color('#333')
ax2.spines['left'].set_color('#333')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

# Add value labels on bars
for i, v in enumerate(results_df['hv_change_pct']):
    ax2.text(i, v + (1 if v > 0 else -2), f'{v}%',
             ha='center', va='bottom', color='white', fontsize=8)

plt.suptitle('Nifty Event Calendar Backtest — RBI 2024-2025',
             fontsize=14, color='white', y=1.02)
plt.tight_layout()
plt.savefig('event_backtest.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0a')
plt.show()

# Summary
crushed = results_df['iv_crushed'].sum()
total = len(results_df)
print(f"\nIV Crush rate: {crushed}/{total} events ({round(crushed/total*100)}%)")
print(f"Average HV change: {round(results_df['hv_change_pct'].mean(), 2)}%")
print(f"Avg crush magnitude: {round(results_df[results_df['iv_crushed']]['hv_change_pct'].mean(), 2)}%")
print(f"Avg spike magnitude: {round(results_df[~results_df['iv_crushed']]['hv_change_pct'].mean(), 2)}%")

In [ ]:
# ============================================
# P&L SIMULATION
# Strategy: Sell 1 ATM Straddle 2 days before
# RBI, buy back 2 days after
# Assumption: Premium = HV * Spot * sqrt(T/252)
# ============================================

pnl_results = []

for date in rbi_dates:
    rbi_date = pd.Timestamp(date).tz_localize('Asia/Kolkata')

    # Get spot price 2 days before RBI
    before_window = hist[hist.index < rbi_date].tail(3)
    after_window = hist[hist.index > rbi_date].head(3)

    if len(before_window) >= 2 and len(after_window) >= 2:
        spot_entry = before_window['Close'].iloc[-2]
        spot_exit = after_window['Close'].iloc[1]

        hv_entry = before_window['HV10'].iloc[-2] / 100
        hv_exit = after_window['HV10'].iloc[1] / 100

        # ATM Straddle premium = 2 * Call price
        # Using simplified BSM: Premium ≈ 0.8 * S * IV * sqrt(T)
        T_entry = 7/252  # 7 days to expiry assumption
        T_exit = 3/252   # 3 days remaining

        premium_sold = 0.8 * spot_entry * hv_entry * (T_entry ** 0.5)
        premium_bought = 0.8 * spot_exit * hv_exit * (T_exit ** 0.5)

        # P&L = premium collected - premium paid to close
        # Also account for delta P&L from spot move
        spot_move = abs(spot_exit - spot_entry)
        delta_loss = spot_move * 0.5  # approximate delta hedge cost

        pnl = premium_sold - premium_bought - delta_loss
        pnl_pct = (pnl / spot_entry) * 100

        pnl_results.append({
            'rbi_date': date,
            'spot_entry': round(spot_entry),
            'spot_exit': round(spot_exit),
            'premium_sold': round(premium_sold),
            'premium_bought': round(premium_bought),
            'spot_move': round(spot_move),
            'pnl': round(pnl),
            'pnl_pct': round(pnl_pct, 3),
            'profitable': pnl > 0
        })

pnl_df = pd.DataFrame(pnl_results)
print(pnl_df[['rbi_date', 'premium_sold', 'premium_bought',
               'spot_move', 'pnl', 'profitable']])
print(f"\nTotal P&L: {round(pnl_df['pnl'].sum())} points")
print(f"Win rate: {pnl_df['profitable'].sum()}/{len(pnl_df)}")
print(f"Avg P&L per trade: {round(pnl_df['pnl'].mean())} points")

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10), facecolor='#0a0a0a')

# Chart 1 - P&L per trade
ax1.set_facecolor('#0a0a0a')
bar_colors = ['#2ECC71' if v else '#E74C3C' for v in pnl_df['profitable']]
bars = ax1.bar(range(len(pnl_df)), pnl_df['pnl'], color=bar_colors, alpha=0.9)
ax1.axhline(y=0, color='white', linewidth=1, alpha=0.5)
ax1.set_xticks(range(len(pnl_df)))
ax1.set_xticklabels(pnl_df['rbi_date'], rotation=45, fontsize=9, color='white')
ax1.set_title('P&L Per Trade | Green = Profit | Red = Loss',
              color='white', fontsize=12)
ax1.set_ylabel('P&L (Nifty Points)', color='white')
ax1.tick_params(colors='white')
ax1.grid(True, alpha=0.2, color='white')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['bottom'].set_color('#333')
ax1.spines['left'].set_color('#333')

for i, v in enumerate(pnl_df['pnl']):
    ax1.text(i, v + (10 if v > 0 else -20), f'{v}',
             ha='center', fontsize=8, color='white')

# Chart 2 - Cumulative P&L
ax2.set_facecolor('#0a0a0a')
cumulative = pnl_df['pnl'].cumsum()
ax2.plot(range(len(cumulative)), cumulative,
         color='#FFD700', linewidth=2, marker='o', markersize=6)
ax2.axhline(y=0, color='white', linewidth=1, alpha=0.5)
ax2.fill_between(range(len(cumulative)), cumulative, 0,
                 where=cumulative >= 0, alpha=0.2, color='#2ECC71')
ax2.fill_between(range(len(cumulative)), cumulative, 0,
                 where=cumulative < 0, alpha=0.2, color='#E74C3C')
ax2.set_xticks(range(len(pnl_df)))
ax2.set_xticklabels(pnl_df['rbi_date'], rotation=45, fontsize=9, color='white')
ax2.set_title('Cumulative P&L — Strategy Equity Curve',
              color='white', fontsize=12)
ax2.set_ylabel('Cumulative P&L (Points)', color='white')
ax2.tick_params(colors='white')
ax2.grid(True, alpha=0.2, color='white')
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['bottom'].set_color('#333')
ax2.spines['left'].set_color('#333')

plt.suptitle('ATM Straddle Short Strategy — RBI Events 2024-2025\nWin Rate: 6/8 (75%) | Total P&L: -86 pts | Tail Risk Demonstrated',
             color='white', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig('pnl_simulation.png', dpi=150, bbox_inches='tight',
            facecolor='#0a0a0a')
plt.show()

## P&L Simulation Results

### Strategy
Sell ATM Straddle 2 days before RBI, buy back 2 days after.

### Results
- Win Rate: 6/8 trades (75%)
- Total P&L: -86 Nifty points
- Best trade: Dec 2024 (+236 points)
- Worst trade: Apr 2025 (-529 points)

### Key Insight — Taleb's Warning in Action
High win rate does not equal profitable strategy.
Two tail events (Feb 2025 rate cut, Apr 2025 continuation)
wiped out 6 consecutive winning trades.
The equity curve peaked at +550 points before collapsing.

### Conclusion
Blind short volatility before RBI = negative expectancy.
The strategy needs:
1. Event filters — avoid when outcome uncertain
2. Position sizing — smaller size on uncertain events  
3. Stop losses — exit if spot moves beyond expected rang